# Run all Evaluations

In [1]:
from pathlib import Path
import pandas as pd
BASE_DIR = Path("results")

Normal ReWOO
                     model  SVAMP  GSM8K
0      ollama/smollm2:360m    0.0   10.0
1      ollama/qwen2.5:0.5b   30.0    0.0
2        ollama/qwen3:0.6b   70.0   20.0
3       ollama/llama3.2:1b    0.0    0.0
4         ollama/gemma3:1b   20.0   10.0
5      ollama/qwen2.5:1.5b   60.0   20.0
6  ollama/deepseek-r1:1.5b   40.0   20.0
7        ollama/qwen3:1.7b   90.0   70.0
8      ollama/smollm2:1.7b   30.0   20.0


ReAct 4 tools (50 samples)
                 model  SVAMP  GSM8K
0  ollama/qwen2.5:0.5b   34.0   10.0
1    ollama/qwen3:0.6b   82.0   50.0
2   ollama/llama3.2:1b   32.0   14.0
3    ollama/qwen3:1.7b   94.0   86.0

ReAct 1 tool (50 samples)
                 model  SVAMP  GSM8K
0  ollama/qwen2.5:0.5b   28.0   14.0
1    ollama/qwen3:0.6b   76.0   62.0
2   ollama/llama3.2:1b   32.0   18.0
3    ollama/qwen3:1.7b   92.0   82.0

In [1]:
from inference.rewoo_test.main import main as rewoo_main
from inference.rewoo_llm_and_tool.main import main as rewoo_llm_and_tool_main
from inference.react4tools.main import main as react4tools_main
from inference.direct_prompting_test.main import main as direct_prompting_main
from inference.react.main import main as react_main 
from inference.direct_prompting_reflection.main import main as direct_prompting_reflection_main
from inference.rewoo_retry.main import main as rewoo_retry_main
from inference.rewoo_retry_v2.main import main as rewoo_retry_v2_main
import os 
import shutil
import logging

logging.basicConfig(level=logging.ERROR)  # Sets default level for logging output

if os.path.exists("results"):
    shutil.rmtree("results")

first_n = 300
# print("ReWOO Retry V2")
# rewoo_retry_v2_main(first_n=first_n)
# print("ReWOO Retry")
# rewoo_retry_main(first_n=first_n)

DP_RFLX =   False
DP      =   True
REWOO   =   True
REACT   =   False

if DP:
    print("Direct Prompting")
    direct_prompting_main(first_n=first_n)
if DP_RFLX:
    print("Direct Prompting Reflection")
    direct_prompting_reflection_main(first_n=first_n)
if REWOO: 
    print("Rewoo")
    rewoo_llm_and_tool_main(first_n=first_n)
if REACT:
    print("React")
    react_main(first_n=first_n)

Using the latest cached version of the dataset since ChilleD/SVAMP couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'default' at C:\Users\patri\.cache\huggingface\datasets\ChilleD___svamp\default\0.0.0\5e0bf1e5e7c0e9c4bc39180d224f41f3f801b7ef (last modified on Wed Sep 10 22:32:19 2025).


Direct Prompting
Limiting to first 300 samples.
✅ ollama/smollm2_135m: Already evaluated all SVAMP test samples. (300/300 samples evaluated)


Using the latest cached version of the dataset since openai/gsm8k couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'main' at C:\Users\patri\.cache\huggingface\datasets\openai___gsm8k\main\0.0.0\e53f048856ff4f594e959d75785d2c2d37b678ee (last modified on Tue Jun 10 14:03:57 2025).


Limiting to first 300 samples.
✅ ollama/smollm2_135m: Already evaluated all GSM8K test samples. (300/300 samples evaluated)
Limiting to first 300 samples.
✅ ollama/gemma3_270m: Already evaluated all SVAMP test samples. (300/300 samples evaluated)
Limiting to first 300 samples.
✅ ollama/gemma3_270m: Already evaluated all GSM8K test samples. (300/300 samples evaluated)
Limiting to first 300 samples.
✅ ollama/smollm2_1.7b: Already evaluated all SVAMP test samples. (300/300 samples evaluated)
Limiting to first 300 samples.
✅ ollama/smollm2_1.7b: Already evaluated all GSM8K test samples. (300/300 samples evaluated)
                 model  SVAMP  GSM8K
0  ollama/smollm2_135m   4.67   0.33
1   ollama/gemma3_270m  26.33  16.67
2  ollama/smollm2_1.7b  56.67  39.00
Rewoo
Limiting to first 300 samples.
✅ ollama/smollm2_135m: Already evaluated all SVAMP test samples. (300/300 samples evaluated)
Limiting to first 300 samples.
✅ ollama/smollm2_135m: Already evaluated all GSM8K test samples. (300/300

## ReWOO

In [3]:
import pandas as pd
import os


dfs = []
for root, dirs, files in os.walk(BASE_DIR / "rewoo_v2"):
    for file in files:
        fpath = os.path.join(root, file)
        df_r = pd.read_excel(fpath)
        df_r["Method"] = "ReWOO"
        df_r["Model"] = Path(fpath).parent.name + "/" + Path(fpath).stem
        df_r["Model"] = df_r["Model"].str.replace("_", ":")
        df_r["Dataset"] = Path(fpath).parent.parent.name
        if Path(fpath).parent.parent.name == "SVAMP":
            df_r["target_answer"] = df_r["Equation"] + " = " + df_r["target_answer"].astype(str)
        
        # select columns  Method", "question", "response", "model_history
        df_r = df_r[["Method", "Model", "Dataset", "question", "target_answer", "response", "model_history", "is_correct", "format_correct", "input_tokens", "output_tokens", "total_tokens", "reasoning"]]
        dfs.append(df_r)

df_rewoo = pd.concat(dfs)
df_rewoo.head()

,Method,Model,Dataset,question,target_answer,response,model_history,is_correct,format_correct,input_tokens,output_tokens,total_tokens,reasoning
0,ReWOO,google/gemma-3-27b-it,SVAMP,Winter is almost here and most animals are mig...,( 62.0 - 35.0 ) = 27,27.0,[{'plan': {'steps': [['Calculate the differenc...,True,True,733.0,0.0,733.0,Plan: Calculate the difference between the num...
1,ReWOO,google/gemma-3-27b-it,SVAMP,Paige raised 7 goldfish and 12 catfish in the ...,( ( 7.0 + 12.0 ) - 15.0 ) = 4,4.0,[{'plan': {'steps': [['Calculate the total num...,True,True,728.0,0.0,728.0,Plan: Calculate the total number of fish Paige...
2,ReWOO,google/gemma-3-27b-it,SVAMP,Marco and his dad went strawberry picking. Tog...,( ( 22.0 - 36.0 ) + 30.0 ) = 16,16.0,"[{'plan': {'steps': [[""Calculate the initial w...",True,True,853.0,0.0,853.0,Plan: Calculate the initial weight of Marco's ...
3,ReWOO,google/gemma-3-27b-it,SVAMP,Debby bought 200 water bottles and 256 soda bo...,( 256.0 / 4.0 ) = 64,64.0,[{'plan': {'steps': [['Calculate the total num...,True,True,746.0,0.0,746.0,Plan: Calculate the total number of soda bottl...
4,ReWOO,google/gemma-3-27b-it,SVAMP,There were 106 dollars in Olivia's wallet. Aft...,( ( 106.0 - 26.0 ) - 49.0 ) = 31,31.0,[{'plan': {'steps': [['Calculate the total amo...,True,True,757.0,0.0,757.0,Plan: Calculate the total amount Olivia spent ...


In [13]:
dfs = []
for root, dirs, files in os.walk(BASE_DIR / "rewoo_retry"):
    for file in files:
        fpath = os.path.join(root, file)
        df_r = pd.read_excel(fpath)
        df_r["Method"] = "ReWOO-Retry"
        df_r["Model"] = Path(fpath).parent.name + "/" + Path(fpath).stem
        df_r["Model"] = df_r["Model"].str.replace("_", ":")
        df_r["Dataset"] = Path(fpath).parent.parent.name
        if Path(fpath).parent.parent.name == "SVAMP":
            df_r["target_answer"] = df_r["Equation"] + " = " + df_r["target_answer"].astype(str)
        
        # select columns  Method", "question", "response", "model_history
        df_r = df_r[["Method", "Model", "Dataset", "question", "target_answer", "response", "is_correct", "format_correct", "input_tokens", "output_tokens", "total_tokens", "reasoning"]]
        dfs.append(df_r)

df_rewoo_retry = pd.concat(dfs)
df_rewoo_retry.head()

,Method,Model,Dataset,question,target_answer,response,is_correct,format_correct,input_tokens,output_tokens,total_tokens,reasoning
0,ReWOO-Retry,ollama/qwen3:1.7b,SVAMP,Winter is almost here and most animals are mig...,( 62.0 - 35.0 ) = 27,"<think>\nOkay, let's see. The problem is about...",True,True,859.0,691.0,1550.0,Plan: Subtract the number of bird families tha...
1,ReWOO-Retry,ollama/qwen3:1.7b,SVAMP,Paige raised 7 goldfish and 12 catfish in the ...,( ( 7.0 + 12.0 ) - 15.0 ) = 4,"<think>\nOkay, let's see. Paige had 7 goldfish...",True,True,845.0,783.0,1628.0,Plan: Calculate the initial number of fish by ...
2,ReWOO-Retry,ollama/qwen3:1.7b,SVAMP,Marco and his dad went strawberry picking. Tog...,( ( 22.0 - 36.0 ) + 30.0 ) = 16,"<think>\nOkay, let's try to figure out this pr...",True,True,874.0,1466.0,2340.0,Plan: Adding 30 to the initial 22 pounds to ge...
3,ReWOO-Retry,ollama/qwen3:1.7b,SVAMP,Debby bought 200 water bottles and 256 soda bo...,( 256.0 / 4.0 ) = 64,"<think>\nOkay, let's see. Debby bought 200 wat...",True,True,822.0,579.0,1401.0,Plan: Divide the total number of soda bottles ...
4,ReWOO-Retry,ollama/qwen3:1.7b,SVAMP,There were 106 dollars in Olivia's wallet. Aft...,( ( 106.0 - 26.0 ) - 49.0 ) = 31,"<think>\nOkay, let's see. Olivia had $106 in h...",True,True,862.0,935.0,1797.0,Plan: Calculate the total amount Olivia spent ...


In [ ]:
dfs = []
for root, dirs, files in os.walk(BASE_DIR / "rewoo_retry_v2"):
    for file in files:
        fpath = os.path.join(root, file)
        df_r = pd.read_excel(fpath)
        df_r["Method"] = "ReWOO-Retry-V2"
        df_r["Model"] = Path(fpath).parent.name + "/" + Path(fpath).stem
        df_r["Model"] = df_r["Model"].str.replace("_", ":")
        df_r["Dataset"] = Path(fpath).parent.parent.name
        if Path(fpath).parent.parent.name == "SVAMP":
            df_r["target_answer"] = df_r["Equation"] + " = " + df_r["target_answer"].astype(str)
        
        # select columns  Method", "question", "response", "model_history
        df_r = df_r[["Method", "Model", "Dataset", "question", "target_answer", "response", "is_correct", "format_correct", "input_tokens", "output_tokens", "total_tokens", "reasoning"]]
        dfs.append(df_r)

df_rewoo_retry_v2 = pd.concat(dfs)
df_rewoo_retry_v2.head()

## DP

In [ ]:
import pandas as pd
import os

dfs = []
for root, dirs, files in os.walk(BASE_DIR / "dp-reflect"):
    for file in files:
        fpath = os.path.join(root, file)
        df_r = pd.read_excel(fpath, sheet_name="Original")
        df_r["Method"] = "DP Reflection"
        df_r["Model"] = Path(fpath).parent.name + "/" + Path(fpath).stem
        df_r["Dataset"] = Path(fpath).parent.parent.name
        # select columns  Method", "question", "response", "model_history
        df_r = df_r[["Method", "Model", "Dataset", "question", "target_answer", "response", "is_correct", "input_tokens", "output_tokens", "total_tokens", "reasoning"]]
        dfs.append(df_r)

df_dp_reflect = pd.concat(dfs)
df_dp_reflect.head()

,Method,Model,Dataset,question,target_answer,response,is_correct,input_tokens,output_tokens,total_tokens,reasoning
0,DP Reflection,ollama/qwen3_1.7b,SVAMP,Winter is almost here and most animals are mig...,27,"<think>\nOkay, let's tackle this problem step ...",True,6362,3892,10254,"ANSWER:\n<think>\nOkay, let's see. So the prob..."
1,DP Reflection,ollama/qwen3_1.7b,SVAMP,Paige raised 7 goldfish and 12 catfish in the ...,4,"<think>\nOkay, let's tackle this problem step ...",True,3123,1667,4790,"ANSWER:\n<think>\nOkay, let's see. Paige had 7..."
2,DP Reflection,ollama/qwen3_1.7b,SVAMP,Marco and his dad went strawberry picking. Tog...,16,"<think>\nOkay, let's tackle this problem step ...",True,5126,3148,8274,"ANSWER:\n<think>\nOkay, let's see. So Marco an..."
3,DP Reflection,ollama/qwen3_1.7b,SVAMP,Debby bought 200 water bottles and 256 soda bo...,64,"<think>\nOkay, let's tackle this problem step ...",True,4154,2462,6616,"ANSWER:\n<think>\nOkay, let's see. Debby bough..."
4,DP Reflection,ollama/qwen3_1.7b,SVAMP,There were 106 dollars in Olivia's wallet. Aft...,31,"<think>\nOkay, let's tackle this problem step ...",True,3386,1938,5324,"ANSWER:\n<think>\nOkay, let's see. Olivia had ..."


In [ ]:
import pandas as pd
import os

dfs = []
for root, dirs, files in os.walk(BASE_DIR / "dp"):
    for file in files:
        fpath = os.path.join(root, file)
        df_r = pd.read_csv(fpath)
        df_r["Method"] = "Direct Prompting"
        df_r["Model"] = Path(fpath).parent.name + "/" + Path(fpath).stem
        df_r["Dataset"] = Path(fpath).parent.parent.name
        # select columns  Method", "question", "response", "model_history
        df_r = df_r[["Method", "Model", "Dataset", "question", "target_answer", "response", "is_correct", "input_tokens", "output_tokens", "total_tokens", "reasoning", "is_correct_llm_judge"]]
        dfs.append(df_r)

df_dp = pd.concat(dfs)
df_dp.head()

,Method,Model,Dataset,question,target_answer,response,is_correct,input_tokens,output_tokens,total_tokens,reasoning
0,Direct Prompting,google/gemma-3-27b-it,SVAMP,Winter is almost here and most animals are mig...,27,Let $A$ be the number of bird families that fl...,True,NaN,NaN,NaN,Let $A$ be the number of bird families that fl...
1,Direct Prompting,google/gemma-3-27b-it,SVAMP,Paige raised 7 goldfish and 12 catfish in the ...,4,Let $G$ be the number of goldfish Paige raised...,True,NaN,NaN,NaN,Let $G$ be the number of goldfish Paige raised...
2,Direct Prompting,google/gemma-3-27b-it,SVAMP,Marco and his dad went strawberry picking. Tog...,16,Let $M$ be the weight of Marco's strawberries ...,False,NaN,NaN,NaN,Let $M$ be the weight of Marco's strawberries ...
3,Direct Prompting,google/gemma-3-27b-it,SVAMP,Debby bought 200 water bottles and 256 soda bo...,64,Let $W$ be the number of water bottles Debby b...,True,NaN,NaN,NaN,Let $W$ be the number of water bottles Debby b...
4,Direct Prompting,google/gemma-3-27b-it,SVAMP,There were 106 dollars in Olivia's wallet. Aft...,31,Let $W$ be the initial amount of money in Oliv...,True,NaN,NaN,NaN,Let $W$ be the initial amount of money in Oliv...


In [ ]:
df_dp.groupby(["Model", "Dataset"]).agg({
    "question": "count",
    "is_correct": "mean",
    "is_correct_llm_judge": "mean",
    "input_tokens": "mean",
    "output_tokens": "mean",
    "total_tokens": "mean"
})

question  is_correct  input_tokens  \
Model                   Dataset                                       
google/gemma-3-27b-it   GSM8K          20        1.00           NaN   
                        SVAMP          20        0.80           NaN   
ollama/deepseek-r1:1.5b GSM8K          20        0.65         64.35   
                        SVAMP          20        0.80         43.90   
ollama/gemma3:1b        GSM8K          20        0.40         70.10   
                        SVAMP          20        0.60         49.45   
ollama/llama3.2:1b      GSM8K          20        0.25         83.95   
                        SVAMP          20        0.55         63.15   
ollama/qwen2-math:1.5b  GSM8K          20        0.70         67.35   
                        SVAMP          20        0.90         46.90   
ollama/qwen2.5:0.5b     GSM8K          20        0.20         90.35   
                        SVAMP          20        0.45         69.90   
ollama/qwen2.5:1.5b     GSM8K          20        0.55         90.35   
                        SVAMP          20        0.85         69.90   
ollama/qwen3:0.6b       GSM8K          20        0.40         69.35   
                        SVAMP          20        0.90         48.90   
ollama/qwen3:1.7b       GSM8K          20        0.55         69.35   
                        SVAMP          20        0.70         48.90   
ollama/smollm2:1.7b     GSM8K          20        0.25         90.90   
                        SVAMP          20        0.75         70.05   
ollama/smollm2:135m     GSM8K          20        0.00         91.90   
                        SVAMP          20        0.00         71.05   
ollama/smollm2:360m     GSM8K          20        0.10         91.90   
                        SVAMP          20        0.25         71.05   

                                 output_tokens  total_tokens  
Model                   Dataset                               
google/gemma-3-27b-it   GSM8K              NaN           NaN  
                        SVAMP              NaN           NaN  
ollama/deepseek-r1:1.5b GSM8K           745.50        809.85  
                        SVAMP           302.65        346.55  
ollama/gemma3:1b        GSM8K           787.55        857.65  
                        SVAMP           790.35        839.80  
ollama/llama3.2:1b      GSM8K           151.30        235.25  
                        SVAMP            91.15        154.30  
ollama/qwen2-math:1.5b  GSM8K           361.20        428.55  
                        SVAMP           195.80        242.70  
ollama/qwen2.5:0.5b     GSM8K           346.25        436.60  
                        SVAMP           207.25        277.15  
ollama/qwen2.5:1.5b     GSM8K           351.00        441.35  
                        SVAMP           175.95        245.85  
ollama/qwen3:0.6b       GSM8K          1646.35       1715.70  
                        SVAMP           995.50       1044.40  
ollama/qwen3:1.7b       GSM8K          1822.35       1891.70  
                        SVAMP          1390.90       1439.80  
ollama/smollm2:1.7b     GSM8K           227.60        318.50  
                        SVAMP            89.75        159.80  
ollama/smollm2:135m     GSM8K           267.45        359.35  
                        SVAMP           128.75        199.80  
ollama/smollm2:360m     GSM8K           197.25        289.15  
                        SVAMP           124.25        195.30

## ReAct

In [17]:
dfs = []
for root, dirs, files in os.walk(BASE_DIR / "react"):
    for file in files:
        fpath = os.path.join(root, file)
        df_r = pd.read_csv(fpath)
        df_r["Method"] = "ReAct"
        df_r["Model"] = Path(fpath).parent.name + "/" + Path(fpath).stem
        df_r["Dataset"] = Path(fpath).parent.parent.name
        # select columns  Method", "question", "response", "model_history
        df_r = df_r[["Method", "Model", "Dataset", "question", "target_answer", "response", "is_correct", "input_tokens", "output_tokens", "total_tokens", "reasoning"]]
        dfs.append(df_r)

df_react = pd.concat(dfs)
df_react.head()

,Method,Model,Dataset,question,target_answer,response,is_correct,input_tokens,output_tokens,total_tokens,reasoning
0,ReAct,ollama/qwen2.5:0.5b,SVAMP,Winter is almost here and most animals are mig...,27,There were 33 more bird families that flew awa...,False,1036,56,1092,[AI]: \n[TOOL]: 33\n[AI]: There were 33 more b...
1,ReAct,ollama/qwen2.5:0.5b,SVAMP,Paige raised 7 goldfish and 12 catfish in the ...,4,Paige lost 4 fish in the pond.,True,977,39,1016,[AI]: \n[TOOL]: 4\n[AI]: Paige lost 4 fish in ...
2,ReAct,ollama/qwen2.5:0.5b,SVAMP,Marco and his dad went strawberry picking. Tog...,16,The expression is not valid. Let's try another...,False,1029,106,1135,[AI]: \n[TOOL]: Error: unmatched ')' (<string>...
3,ReAct,ollama/qwen2.5:0.5b,SVAMP,Debby bought 200 water bottles and 256 soda bo...,64,Debby would not be able to drink all the soda ...,False,1007,116,1123,[AI]: \n[TOOL]: -14.0\n[AI]: Debby would not b...
4,ReAct,ollama/qwen2.5:0.5b,SVAMP,There were 106 dollars in Olivia's wallet. Aft...,31,Olivia spent 28.5 dollars at the supermarket.,False,1003,43,1046,[AI]: \n[TOOL]: 28.5\n[AI]: Olivia spent 28.5 ...


## ALL

In [ ]:
df = pd.concat([df_rewoo, df_rewoo_retry, df_rewoo_retry_v2, df_dp, df_react, df_dp_reflect])
print(df["Method"].unique())
df.head()

['ReWOO' 'ReWOO-Retry' 'Direct Prompting' 'ReAct' 'DP Reflection']


,Method,Model,Dataset,question,target_answer,response,model_history,is_correct,format_correct,input_tokens,output_tokens,total_tokens,reasoning
0,ReWOO,google/gemma-3-27b-it,SVAMP,Winter is almost here and most animals are mig...,( 62.0 - 35.0 ) = 27,27.0,[{'plan': {'steps': [['Calculate the differenc...,True,True,733.0,0.0,733.0,Plan: Calculate the difference between the num...
1,ReWOO,google/gemma-3-27b-it,SVAMP,Paige raised 7 goldfish and 12 catfish in the ...,( ( 7.0 + 12.0 ) - 15.0 ) = 4,4.0,[{'plan': {'steps': [['Calculate the total num...,True,True,728.0,0.0,728.0,Plan: Calculate the total number of fish Paige...
2,ReWOO,google/gemma-3-27b-it,SVAMP,Marco and his dad went strawberry picking. Tog...,( ( 22.0 - 36.0 ) + 30.0 ) = 16,16.0,"[{'plan': {'steps': [[""Calculate the initial w...",True,True,853.0,0.0,853.0,Plan: Calculate the initial weight of Marco's ...
3,ReWOO,google/gemma-3-27b-it,SVAMP,Debby bought 200 water bottles and 256 soda bo...,( 256.0 / 4.0 ) = 64,64.0,[{'plan': {'steps': [['Calculate the total num...,True,True,746.0,0.0,746.0,Plan: Calculate the total number of soda bottl...
4,ReWOO,google/gemma-3-27b-it,SVAMP,There were 106 dollars in Olivia's wallet. Aft...,( ( 106.0 - 26.0 ) - 49.0 ) = 31,31.0,[{'plan': {'steps': [['Calculate the total amo...,True,True,757.0,0.0,757.0,Plan: Calculate the total amount Olivia spent ...


# Plot 1: Accuracy

In [19]:
# Assume 'grouped' is your DataFrame with columns: Model, Method, Dataset, mean_correct_answers

# If 'grouped' does not have 'Dataset', use the one grouped by ["Dataset", "Model", "Method"]
grouped = df.groupby(["Dataset", "Model", "Method"]).agg(
    mean_correct_answers=("is_correct", "mean")
).reset_index()

# Pivot the table to get the desired format
pivot = grouped.pivot_table(
    index="Model",
    columns=["Dataset", "Method"],
    values="mean_correct_answers"
)

# Optional: Convert to percentage and round
pivot = (pivot * 100).round(0).astype("Int64").astype(str) + "%"

# Reorder columns if needed
pivot = pivot.reindex(
    columns=pd.MultiIndex.from_product(
        [["SVAMP", "GSM8K"], ["Direct Prompting","DP Reflection", "ReAct", "ReWOO", "ReWOO-Retry"]]
    ),
    fill_value=""
)

# Display the table
display(pivot)

SVAMP                              \
                        Direct Prompting DP Reflection  ReAct  ReWOO   
Model                                                                  
google/gemma-3-27b-it                80%         <NA>%  <NA>%    90%   
ollama/deepseek-r1:1.5b              80%         <NA>%  <NA>%    85%   
ollama/deepseek-r1_1.5b            <NA>%           70%  <NA>%  <NA>%   
ollama/gemma3:1b                     60%         <NA>%  <NA>%    25%   
ollama/llama3.2:1b                   55%         <NA>%    30%    20%   
ollama/qwen2-math:1.5b               90%         <NA>%  <NA>%     0%   
ollama/qwen2.5:0.5b                  45%         <NA>%    30%    20%   
ollama/qwen2.5:1.5b                  85%         <NA>%  <NA>%  <NA>%   
ollama/qwen2.5_0.5b                <NA>%           30%  <NA>%  <NA>%   
ollama/qwen3:0.6b                    90%         <NA>%    80%    60%   
ollama/qwen3:1.7b                    70%         <NA>%    90%    90%   
ollama/qwen3_0.6b                  <NA>%           95%  <NA>%  <NA>%   
ollama/qwen3_1.7b                  <NA>%           95%  <NA>%  <NA>%   
ollama/smollm2:1.7b                  75%         <NA>%  <NA>%  <NA>%   
ollama/smollm2:135m                   0%         <NA>%  <NA>%  <NA>%   
ollama/smollm2:360m                  25%         <NA>%  <NA>%    10%   
ollama/smollm2_360m                <NA>%           15%  <NA>%  <NA>%   

                                               GSM8K                       \
                        ReWOO-Retry Direct Prompting DP Reflection  ReAct   
Model                                                                       
google/gemma-3-27b-it         <NA>%             100%         <NA>%  <NA>%   
ollama/deepseek-r1:1.5b          0%              65%         <NA>%  <NA>%   
ollama/deepseek-r1_1.5b       <NA>%            <NA>%           50%  <NA>%   
ollama/gemma3:1b              <NA>%              40%         <NA>%  <NA>%   
ollama/llama3.2:1b            <NA>%              25%         <NA>%     5%   
ollama/qwen2-math:1.5b        <NA>%              70%         <NA>%  <NA>%   
ollama/qwen2.5:0.5b              0%              20%         <NA>%    15%   
ollama/qwen2.5:1.5b           <NA>%              55%         <NA>%  <NA>%   
ollama/qwen2.5_0.5b           <NA>%            <NA>%           20%  <NA>%   
ollama/qwen3:0.6b               75%              40%         <NA>%    55%   
ollama/qwen3:1.7b               85%              55%         <NA>%    80%   
ollama/qwen3_0.6b             <NA>%            <NA>%           50%  <NA>%   
ollama/qwen3_1.7b             <NA>%            <NA>%           85%  <NA>%   
ollama/smollm2:1.7b           <NA>%              25%         <NA>%  <NA>%   
ollama/smollm2:135m           <NA>%               0%         <NA>%  <NA>%   
ollama/smollm2:360m           <NA>%              10%         <NA>%  <NA>%   
ollama/smollm2_360m           <NA>%            <NA>%            5%  <NA>%   

                                            
                         ReWOO ReWOO-Retry  
Model                                       
google/gemma-3-27b-it      75%       <NA>%  
ollama/deepseek-r1:1.5b    65%          0%  
ollama/deepseek-r1_1.5b  <NA>%       <NA>%  
ollama/gemma3:1b           20%       <NA>%  
ollama/llama3.2:1b         10%       <NA>%  
ollama/qwen2-math:1.5b      0%       <NA>%  
ollama/qwen2.5:0.5b        10%          0%  
ollama/qwen2.5:1.5b      <NA>%       <NA>%  
ollama/qwen2.5_0.5b      <NA>%       <NA>%  
ollama/qwen3:0.6b          35%         35%  
ollama/qwen3:1.7b          75%         45%  
ollama/qwen3_0.6b        <NA>%       <NA>%  
ollama/qwen3_1.7b        <NA>%       <NA>%  
ollama/smollm2:1.7b      <NA>%       <NA>%  
ollama/smollm2:135m      <NA>%       <NA>%  
ollama/smollm2:360m         0%       <NA>%  
ollama/smollm2_360m      <NA>%       <NA>%

## Statistical Significance

In [ ]:
from statistical_significance import compare_accuracies

compare_accuracies(0.40, 0.50, 100)

# Plot 2: Token Usage

## Plot 2a: Mean Token Usage

In [ ]:
# Group by Model and Method, and calculate mean tokens
mean_token_usage = df.groupby(["Model", "Method"]).agg(
    mean_correct_answers=("is_correct", "mean"),
    mean_input_tokens=("input_tokens", "mean"),
    mean_output_tokens=("output_tokens", "mean"),
    mean_total_tokens=("total_tokens", "mean"),
)
mean_token_usage

## Plot 2b: Mean Token Usage SVAMP

In [ ]:
svamp_df = df[df["Dataset"] == "SVAMP"]
token_usage_svamp = svamp_df.groupby(["Model", "Method"]).agg(
    mean_correct_answers=("is_correct", "mean"),
    mean_input_tokens=("input_tokens", "mean"),
    mean_output_tokens=("output_tokens", "mean"),
    mean_total_tokens=("total_tokens", "mean"),
)
token_usage_svamp

## Plot 2c: Mean Token Usage GSM8K

In [ ]:
gsm8k_df = df[df["Dataset"] == "GSM8K"]
token_usage_gsm8K = gsm8k_df.groupby(["Model", "Method"]).agg(
    mean_correct_answers=("is_correct", "mean"),
    mean_input_tokens=("input_tokens", "mean"),
    mean_output_tokens=("output_tokens", "mean"),
    mean_total_tokens=("total_tokens", "mean"),
)
token_usage_gsm8K

In [ ]:
# Create melted DataFrame for token usage visualization
import pandas as pd

# Add dataset labels to each DataFrame
mean_token_usage_labeled = mean_token_usage.copy()
mean_token_usage_labeled['Dataset'] = 'ALL'

token_usage_svamp_labeled = token_usage_svamp.copy()
token_usage_svamp_labeled['Dataset'] = 'SVAMP'

token_usage_gsm8K_labeled = token_usage_gsm8K.copy()
token_usage_gsm8K_labeled['Dataset'] = 'GSM8K'

# Combine all token usage DataFrames
combined_token_usage = pd.concat([
    mean_token_usage_labeled,
    token_usage_svamp_labeled, 
    token_usage_gsm8K_labeled
]).reset_index()

# Melt the DataFrame to long format for plotting
melted = combined_token_usage.melt(
    id_vars=['Model', 'Method', 'Dataset'],
    value_vars=['mean_input_tokens', 'mean_output_tokens', 'mean_total_tokens'],
    var_name='TokenType',
    value_name='MeanTokens'
)

print("Melted DataFrame structure:")
print(melted.head())
print(f"\nShape: {melted.shape}")
print(f"Unique TokenTypes: {melted['TokenType'].unique()}")
print(f"Unique Datasets: {melted['Dataset'].unique()}")

## Plot 2d: All together

In [ ]:
# Stacked barplot: input and output tokens composition by Method (ALL datasets aggregated)
import matplotlib.pyplot as plt

# Filter for ALL dataset and only input/output tokens
agg_data = melted[(melted['Dataset'] == 'ALL') & (melted['TokenType'].isin(['mean_input_tokens', 'mean_output_tokens']))].copy()

# Aggregate by Method
agg_pivot = agg_data.pivot_table(
    index='Method',
    columns='TokenType',
    values='MeanTokens',
    aggfunc='mean'
).reset_index()

plt.figure(figsize=(7, 5))
plt.bar(agg_pivot['Method'], agg_pivot['mean_input_tokens'], label='Input Tokens')
plt.bar(agg_pivot['Method'], agg_pivot['mean_output_tokens'],
        bottom=agg_pivot['mean_input_tokens'], label='Output Tokens')
plt.title('Input/Output Token Composition by Method (ALL)')
plt.ylabel('Mean Tokens')
plt.xlabel('Method')
plt.legend()
plt.tight_layout()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Calculate mean accuracy and mean total tokens per Model & Method (aggregated across datasets or select one dataset)
summary = df.groupby(["Dataset", "Model", "Method"]).agg(
    mean_accuracy=("is_correct", "mean"),
    mean_total_tokens=("total_tokens", "mean")
).reset_index()

summary["mean_accuracy"] *= 100  # convert to %

# Example: Plot for GSM8K dataset
gsm8k = summary[summary["Dataset"] == "GSM8K"]

plt.figure(figsize=(10, 6))
sns.scatterplot(data=gsm8k, x="mean_total_tokens", y="mean_accuracy", hue="Method", style="Model", s=100)

plt.title("Accuracy vs. Token Cost (GSM8K Dataset)")
plt.xlabel("Mean Total Tokens")
plt.ylabel("Accuracy (%)")
plt.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0.)
plt.grid(True)
plt.tight_layout()
plt.show()

# Plot 3: Token Usage per Is-Correct

In [ ]:
# Table: Mean output tokens aggregated by Method and is_correct (ALL datasets)
agg_output = df.groupby(['Method', 'is_correct'])['output_tokens'].mean().unstack()
agg_output = agg_output.round(1)
agg_output.columns = ['Incorrect', 'Correct'] if 0 in agg_output.columns else agg_output.columns
display(agg_output)

In [ ]:
# plot agg_output as barplot
agg_output.plot(kind='bar', figsize=(10, 6))
plt.title('Mean Output Tokens by Method and Is-Correct')
plt.xlabel('Method')
plt.ylabel('Mean Output Tokens')
plt.xticks(rotation=45)
plt.legend(title='Is-Correct')
plt.tight_layout()
plt.show()

In [ ]:
import re

def has_wrong_calculation(reasoning_text):
    """
    Detect if reasoning text contains wrongly calculated equations.
    Returns True if wrong calculations are found, False otherwise.
    
    Args:
        reasoning_text (str): The reasoning text to analyze
    
    Returns:
        bool: True if wrong calculations detected, False otherwise
    """
    if not isinstance(reasoning_text, str):
        return False
    
    # Pattern to match equations like "1+1=3", "5*2=11", "10-3=6", "8/2=3"
    equation_pattern = r'(\d+(?:\.\d+)?)\s*([+\-*/])\s*(\d+(?:\.\d+)?)\s*=\s*(\d+(?:\.\d+)?)'
    
    equations = re.findall(equation_pattern, reasoning_text)
    
    for equation in equations:
        num1, operator, num2, result = equation
        num1, num2, result = float(num1), float(num2), float(result)
        
        # Calculate the correct result
        if operator == '+':
            correct_result = num1 + num2
        elif operator == '-':
            correct_result = num1 - num2
        elif operator == '*':
            correct_result = num1 * num2
        elif operator == '/':
            if num2 == 0:
                continue  # Skip division by zero
            correct_result = num1 / num2
        else:
            continue
        
        # Check if the stated result is wrong (with small tolerance for floating point)
        if abs(correct_result - result) > 1e-6:
            return True
    
    return False

# Test the function
test_cases = [
    "1+1=3 and 1+1=2",  # Wrong
    "1+1=2 and 1+1=2",  # Correct
    "5*2=10 and 5*1=6", # Wrong
    "5*2=10", # Correct
    "10-3=6", # Wrong
    "10-3=7", # Correct
    "8/2=3",  # Wrong
    "8/2=4",  # Correct
    "The answer is 5+3=8 which is correct",  # Correct
    "We calculate 2+2=5 so the total is 5",  # Wrong
    "No equations here"  # No equations
]

print("Testing wrong calculation detection:")
for test in test_cases:
    result = has_wrong_calculation(test)
    print(f"'{test}' -> {result}")

In [ ]:
# Apply wrong calculation detection to the dataset
df_t = df_dp.copy()
df_t['has_wrong_calc'] = df_t['reasoning'].apply(has_wrong_calculation)

In [ ]:
# Show summary by model
wrong_calc_summary = df_t.groupby(["Model", 'has_wrong_calc']).size().unstack(fill_value=0)
wrong_calc_summary['total'] = wrong_calc_summary.sum(axis=1)
wrong_calc_summary['wrong_calc_rate'] = (wrong_calc_summary[True] / wrong_calc_summary['total'] * 100).round(2)

print("Wrong Calculation Detection Summary:")
wrong_calc_summary

In [ ]:
# Add column for wrong calculation leading to correct result
df_t['wrong_calc_correct_result'] = df_t['has_wrong_calc'] & df_t['is_correct']

# Enhanced summary by model
enhanced_summary = df_t.groupby(["Model"]).agg({
    'has_wrong_calc': 'sum',
    'is_correct': 'sum', 
    'wrong_calc_correct_result': 'sum'
}).assign(
    total=lambda x: len(df_t.groupby("Model").size()),
    wrong_calc_rate=lambda x: (x['has_wrong_calc'] / len(df_t.groupby("Model").size().iloc[0:len(x)]) * 100).round(2),
    correct_rate=lambda x: (x['is_correct'] / len(df_t.groupby("Model").size().iloc[0:len(x)]) * 100).round(2),
    wrong_calc_but_correct_rate=lambda x: (x['wrong_calc_correct_result'] / x['has_wrong_calc'] * 100).fillna(0).round(2)
)

# Reset the total calculation properly
model_counts = df_t.groupby("Model").size()
enhanced_summary['total'] = enhanced_summary.index.map(model_counts)
enhanced_summary['wrong_calc_rate'] = (enhanced_summary['has_wrong_calc'] / enhanced_summary['total'] * 100).round(2)
enhanced_summary['correct_rate'] = (enhanced_summary['is_correct'] / enhanced_summary['total'] * 100).round(2)
enhanced_summary['wrong_calc_but_correct_rate'] = (enhanced_summary['wrong_calc_correct_result'] / enhanced_summary['has_wrong_calc'] * 100).fillna(0).round(2)

print("Enhanced Wrong Calculation Analysis:")
enhanced_summary[['total', 'has_wrong_calc', 'wrong_calc_rate', 'is_correct', 'correct_rate', 'wrong_calc_correct_result', 'wrong_calc_but_correct_rate']]

In [ ]:
# Show examples of wrong calculations that led to correct results
print("Examples of WRONG calculations that led to CORRECT final results:")
wrong_calc_correct_examples = df_t[df_t['wrong_calc_correct_result'] == True][['Model', 'reasoning', 'is_correct', 'target_answer']].head(3)
if len(wrong_calc_correct_examples) > 0:
    for idx, row in wrong_calc_correct_examples.iterrows():
        print(f"\nModel: {row['Model']}")
        print(f"Final Result Correct: {row['is_correct']}")
        print(f"Target Answer: {row['target_answer']}")
        print(f"Reasoning: {row['reasoning']}...")
        print("-" * 80)
else:
    print("No examples found where wrong calculations led to correct results.")

print("\n" + "="*80)
print("Examples of WRONG calculations that led to INCORRECT final results:")
wrong_calc_incorrect_examples = df_t[(df_t['has_wrong_calc'] == True) & (df_t['is_correct'] == False)][['Model', 'reasoning', 'is_correct', 'target_answer']].head(3)
if len(wrong_calc_incorrect_examples) > 0:
    for idx, row in wrong_calc_incorrect_examples.iterrows():
        print(f"\nModel: {row['Model']}")
        print(f"Final Result Correct: {row['is_correct']}")
        print(f"Target Answer: {row['target_answer']}")
        print(f"Reasoning: {row['reasoning']}...")
        print("-" * 80)
else:
    print("No examples found where wrong calculations led to incorrect results.")

# SFT

In [ ]:
records = [
    # {
    #     "Model": "Qwen/Qwen2.5-0.5B-Instruct",
    #     "SFT": False,
    #     "SVAMP": 0.33,
    #     "GSM8K": 0.15
    # },
    {
        "Model": "Qwen/Qwen2.5-0.5B-Instruct",
        "Method": "ReWOO",
        "SFT": True,
        "SVAMP": 50.0,
        "GSM8K": 32.0
    },
    # {
    #     "Model": "Qwen/Qwen3-0.6B",
    #     "SFT": False,
    #     "SVAMP": 6.0,
    #     "GSM8K": 2.0
    # },
    {
        "Model": "Qwen/Qwen3-0.6B",
        "Method": "ReWOO",
        "SFT": True,
        "SVAMP": 70.0,
        "GSM8K": 42.0
    },
    # {
    #     "Model": "Qwen/Qwen3-0.6B",
    #     "Method": "DP",
    #     "SFT": True,
    #     "SVAMP": 72.0,
    #     "GSM8K": 42.0
    # },
    {
        "Model": "HuggingFaceTB/SmolLM2-360M-Instruct",
        "Method": "ReWOO",
        "SFT": True,
        "SVAMP": 12.0,
        "GSM8K": 8.0
    },
]

df_sft = pd.DataFrame(records)
# plotbarplot for SFT
import matplotlib.pyplot as plt

# Set plot style
plt.style.use("ggplot")

# Create the bar plot
fig, ax = plt.subplots(figsize=(10, 6))
df_sft.set_index("Model")[["SVAMP", "GSM8K"]].plot(kind="bar", ax=ax)

# Customize plot
ax.set_ylabel("Accuracy (%)")
ax.set_title("SFT Model Performance on SVAMP and GSM8K")
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
# Show the plot
plt.show()